# The Ops Committee - TRL SFT Training

This notebook builds a multi-level synthetic training corpus from the OpenEnv environment, fine-tunes a small causal LM with TRL `SFTTrainer`, and compares the trained-style committee behavior against a random approval baseline. In Colab, run the cells top to bottom after uploading or cloning this repository.

In [ ]:
!pip -q install -U trl transformers accelerate datasets peft
# Optional GPU path if available for your model/runtime choice:
# !pip -q install -U unsloth

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from training.train_trl_colab import build_sft_dataset, to_chatml_text
from training.eval import run_eval

raw_dataset = build_sft_dataset(n_episodes=200)
len(raw_dataset), raw_dataset[0].keys()

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([
    {"text": to_chatml_text(example)} for example in raw_dataset
])
split = dataset.train_test_split(test_size=0.1, seed=42)
split

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

model_name = "sshleifer/tiny-gpt2"  # fast CPU smoke test; swap to Qwen/Qwen2.5-0.5B-Instruct on GPU
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

args = TrainingArguments(
    output_dir="artifacts/trl_sft_ops_committee",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    dataset_text_field="text",
    max_seq_length=1024,
)
train_result = trainer.train()
train_result

In [ ]:
import csv
from pathlib import Path

Path("artifacts").mkdir(exist_ok=True)
history = trainer.state.log_history
with open("artifacts/training_loss.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["step", "loss"])
    writer.writeheader()
    for row in history:
        if "loss" in row:
            writer.writerow({"step": row.get("step", 0), "loss": row["loss"]})

eval_result = trainer.evaluate()
eval_result

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

loss_df = pd.read_csv("artifacts/training_loss.csv")
if loss_df.empty:
    raise ValueError("No loss rows found in artifacts/training_loss.csv")

plt.figure(figsize=(8, 4.5))
plt.plot(loss_df["step"], loss_df["loss"], marker="o", linewidth=2)
plt.title("Ops Committee SFT Training Loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("artifacts/training_loss.png", dpi=160)
plt.show()

In [ ]:
evidence = run_eval()
evidence